In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("brunogrisci/breast-cancer-gene-expression-cumida")

print("Path to dataset files:", path)

100%|██████████| 61.5M/61.5M [00:00<00:00, 73.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/brunogrisci/breast-cancer-gene-expression-cumida/versions/2


In [ ]:
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


cp: cannot stat '/content/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!kaggle datasets download -d brunogrisci/breast-cancer-gene-expression-cumida --unzip


Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 1741, in dataset_download_cli
    with self.build_kaggle_client() as kaggle:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 688, in build_kaggle_client
    username=self.config_values['username'],
             ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'username'


In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import f_classif, SelectKBest
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, accuracy_score

# ✅ Step 1: Load Dataset
file_path = "/content/Breast_GSE45827.csv"
df = pd.read_csv(file_path)

# Drop 'samples' column if it exists
if 'samples' in df.columns:
    df = df.drop(columns=['samples'])

# ✅ Step 2: Encode Cancer Subtypes
label_encoder = LabelEncoder()
df['type'] = label_encoder.fit_transform(df['type'])

# ✅ Step 3: Balance the Dataset Using SMOTE (Fix for Basal & Luminal B)
X = df.drop(columns=['type'])
y = df['type']

smote = SMOTE(sampling_strategy="auto", random_state=42)  # Automatically balances all classes
X_balanced, y_balanced = smote.fit_resample(X, y)

# ✅ Step 4: Feature Selection (ANOVA F-test - Top 500 Genes)
selector = SelectKBest(score_func=f_classif, k=500)
X_selected = selector.fit_transform(X_balanced, y_balanced)

# Get selected feature names
selected_features = X.columns[selector.get_support()]

# ✅ Step 5: Normalize the feature data *after* selecting the top 500 genes
scaler = StandardScaler()
X_selected_scaled = scaler.fit_transform(X_selected)

# ✅ Step 6: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_selected_scaled, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced)

# ✅ Step 7: Train an Improved Random Forest Model
rf_model = RandomForestClassifier(
    n_estimators=300,  # Increased trees for better generalization
    max_depth=12,  # Increased depth for more complex patterns
    min_samples_split=8,  # Reduced to allow better splits
    min_samples_leaf=4,  # Adjusted for better leaf nodes
    class_weight="balanced_subsample",  # Better balancing of rare subtypes
    random_state=42,
    n_jobs=-1  # Uses all CPU cores for faster training
)
rf_model.fit(X_train, y_train)

# ✅ Step 8: Model Evaluation
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
class_report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)

print("\n✅ Model Evaluation Results:")
print(f"Accuracy: {accuracy:.4f}")
print("\n📊 Classification Report:")
print(class_report)

# ✅ Step 9: Save the Trained Model
model_data = {
    "model": rf_model,
    "feature_names": list(selected_features),
    "accuracy": accuracy  # Save accuracy for reference
}

with open("trained_model_optimized.pkl", "wb") as model_file:
    pickle.dump(model_data, model_file)

with open("scaler_optimized.pkl", "wb") as scaler_file:
    pickle.dump(scaler, scaler_file)

with open("label_encoder_optimized.pkl", "wb") as label_encoder_file:
    pickle.dump(label_encoder, label_encoder_file)

print("\n✅ Model, Scaler, Label Encoder, and Feature Names saved successfully!")

# ✅ Step 10: Save Cross-Validation & Evaluation Results
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, n_jobs=-1)
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

cv_results_df = pd.DataFrame({
    "CV Mean Accuracy": [cv_mean],
    "CV Accuracy Std Dev": [cv_std],
    "Final Accuracy": [accuracy],
    "Number of Features Before": [X.shape[1]],
    "Number of Features After": [X_selected.shape[1]]
})

cv_results_df.to_csv("cross_validation_results.csv", index=False)
print("\n📊 Updated Cross-Validation Results:")
print(cv_results_df)

# ✅ Provide a download link for results
from google.colab import files
files.download("cross_validation_results.csv")


✅ Model Evaluation Results:
Accuracy: 0.9400

📊 Classification Report:
              precision    recall  f1-score   support

         HER       0.89      1.00      0.94         8
       basal       1.00      0.88      0.93         8
   cell_line       1.00      1.00      1.00         8
   luminal_A       0.82      1.00      0.90         9
   luminal_B       1.00      0.78      0.88         9
      normal       1.00      1.00      1.00         8

    accuracy                           0.94        50
   macro avg       0.95      0.94      0.94        50
weighted avg       0.95      0.94      0.94        50


✅ Model, Scaler, Label Encoder, and Feature Names saved successfully!

📊 Updated Cross-Validation Results:
   CV Mean Accuracy  CV Accuracy Std Dev  Final Accuracy  \
0          0.959103              0.02061            0.94   

   Number of Features Before  Number of Features After  
0                      54675                       500  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, f_classif

# ✅ Load Dataset
file_path = "/content/Breast_GSE45827.csv"
df = pd.read_csv(file_path)

# Drop 'samples' column if it exists
if 'samples' in df.columns:
    df = df.drop(columns=['samples'])

# ✅ Encode Cancer Subtypes
label_encoder = LabelEncoder()
df['type'] = label_encoder.fit_transform(df['type'])

# ✅ Keep Only Luminal B & Basal for the Second Model
df_filtered = df[df['type'].isin([label_encoder.transform(['luminal_B'])[0], label_encoder.transform(['basal'])[0]])]

# ✅ Balance the Dataset Using SMOTE
X = df_filtered.drop(columns=['type'])
y = df_filtered['type']
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X, y)

# ✅ Feature Selection (Select Top 500 Genes)
selector = SelectKBest(score_func=f_classif, k=500)
X_selected = selector.fit_transform(X_balanced, y_balanced)
selected_features = X.columns[selector.get_support()]

# ✅ Normalize Features
scaler = StandardScaler()
X_selected_scaled = scaler.fit_transform(X_selected)

# ✅ Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_selected_scaled, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced)

# ✅ Train Luminal B & Basal Model
rf_model_basal_luminalB = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_model_basal_luminalB.fit(X_train, y_train)

# ✅ Evaluate Model Performance
cv_scores = cross_val_score(rf_model_basal_luminalB, X_train, y_train, cv=5, n_jobs=-1)
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

# ✅ Save the Trained Second Model
model_data = {
    "model": rf_model_basal_luminalB,
    "feature_names": list(selected_features)
}

with open("/content/trained_model_basal_luminalB.pkl", "wb") as model_file:
    pickle.dump(model_data, model_file)

with open("/content/scaler_basal_luminalB.pkl", "wb") as scaler_file:
    pickle.dump(scaler, scaler_file)

with open("/content/label_encoder_basal_luminalB.pkl", "wb") as label_encoder_file:
    pickle.dump(label_encoder, label_encoder_file)

print("✅ Second Model (Luminal B & Basal) Saved Successfully!")

✅ Second Model (Luminal B & Basal) Saved Successfully!


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Streamlit App Title
st.title("🧬 Personalized CRISPR Therapy Model")
st.write("Upload your breast cancer gene expression dataset to analyze and get CRISPR-based recommendations.")

# Step 1: Load Pre-trained Model & Feature Names
@st.cache_resource
def load_model():
    try:
        with open("trained_model_optimized.pkl", "rb") as model_file:
            model_data = pickle.load(model_file)
            model = model_data["model"]
            feature_names = model_data["feature_names"]
        with open("scaler_optimized.pkl", "rb") as scaler_file:
            scaler = pickle.load(scaler_file)
        with open("label_encoder_optimized.pkl", "rb") as label_encoder_file:
            label_encoder = pickle.load(label_encoder_file)
        return model, scaler, label_encoder, feature_names
    except FileNotFoundError:
        st.error("🚨 Error: Trained model files not found. Please ensure they are uploaded before running the app.")
        return None, None, None, None

# Load model, scaler, and feature names
model, scaler, label_encoder, feature_names = load_model()

# Expanded list of affected genes per subtype
gene_info = {
    "HER": [
        {"Gene": "ERBB2", "Function": "Encodes HER2 receptor", "Effect": "Overexpressed in HER2+ cancers", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "GRB7", "Function": "Growth signaling protein", "Effect": "Drives aggressive tumor growth", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "PIK3CA", "Function": "Cell signaling enzyme", "Effect": "Mutated in HER2-positive cases", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "FOXA1", "Function": "Transcription factor", "Effect": "Regulates HER2 gene expression", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "PTEN", "Function": "Tumor suppressor", "Effect": "Loss leads to HER2 activation", "CRISPR Recommendation": "Activate with CRISPRa"},
    ],
    "luminal_A": [
        {"Gene": "ESR1", "Function": "Estrogen receptor", "Effect": "Promotes estrogen-dependent growth", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "PGR", "Function": "Progesterone receptor", "Effect": "Regulates hormone response", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "GATA3", "Function": "Transcription factor", "Effect": "Regulates luminal differentiation", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "AR", "Function": "Androgen receptor", "Effect": "Involved in hormonal response", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "BRCA2", "Function": "DNA repair", "Effect": "Deficiency increases risk", "CRISPR Recommendation": "Activate with CRISPRa"},
    ],
    "luminal_B": [
        {"Gene": "CCND1", "Function": "Cell cycle regulator", "Effect": "Overactive in luminal B", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "FOXA1", "Function": "Transcription factor", "Effect": "Regulates estrogen response", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "MYB", "Function": "Regulates proliferation", "Effect": "Upregulated in Luminal B", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "BRCA1", "Function": "DNA repair", "Effect": "Deficiency can lead to cancer", "CRISPR Recommendation": "Activate with CRISPRa"},
    ],
    "basal": [
        {"Gene": "TP53", "Function": "Tumor suppressor", "Effect": "Mutated in basal cancers", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "EGFR", "Function": "Growth receptor", "Effect": "Drives basal-like breast cancer", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "RB1", "Function": "Cell cycle regulator", "Effect": "Lost in basal subtypes", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "BRCA1", "Function": "DNA repair protein", "Effect": "Loss leads to basal cancer", "CRISPR Recommendation": "Activate with CRISPRa"},
    ],
    "normal": [
        {"Gene": "GATA3", "Function": "Transcription factor", "Effect": "Maintains normal tissue identity", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "BRCA1", "Function": "DNA repair protein", "Effect": "Prevents genetic damage", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "FOXA1", "Function": "Transcription factor", "Effect": "Regulates hormonal genes", "CRISPR Recommendation": "Activate with CRISPRa"},
    ],
    "cell_line": [
        {"Gene": "MYC", "Function": "Oncogene", "Effect": "Drives cell proliferation", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "TERT", "Function": "Telomerase enzyme", "Effect": "Maintains unlimited growth", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "CDK4", "Function": "Cell cycle kinase", "Effect": "Promotes rapid division", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "CCNE1", "Function": "Cell cycle regulator", "Effect": "Amplified in aggressive cancers", "CRISPR Recommendation": "Suppress with CRISPRi"},
    ]
}

if model is not None:
    st.success("✅ Model Loaded Successfully!")

    # Step 2: Upload a Patient's Gene Expression Data (Without 'type' column)
    st.subheader("🧪 Upload a Patient's Gene Expression Data")
    patient_file = st.file_uploader("Upload Patient CSV", type=["csv"], key="patient_data")

    if patient_file is not None:
        patient_df = pd.read_csv(patient_file)

        if 'type' in patient_df.columns:
            st.error("🚨 Error: The uploaded patient file should not contain a 'type' column.")
        else:
            st.write("Patient Data Preview:")
            st.dataframe(patient_df.head())

            missing_cols = set(feature_names) - set(patient_df.columns)
            if missing_cols:
                st.error(f"🚨 Error: The uploaded patient file is missing columns: {missing_cols}")
            else:
                patient_df = patient_df[feature_names]

                patient_scaled = scaler.transform(patient_df)
                prediction = model.predict(patient_scaled)
                predicted_label = label_encoder.inverse_transform(prediction)[0]

                st.success(f"🩺 *Predicted Cancer Subtype: {predicted_label}*")

                st.subheader("🧬 Affected Genes in Patient Sample")
                if predicted_label in gene_info:
                    affected_genes_df = pd.DataFrame(gene_info[predicted_label])
                    st.dataframe(affected_genes_df)
                else:
                    st.warning("⚠ No specific gene information available for this subtype.")

                st.subheader("🧬 Recommended CRISPR Gene Alteration")
                st.info("🔬 Based on affected genes, use CRISPRi (Gene Suppression) or CRISPRa (Gene Activation).")
                # Explanation of how CRISPRi/CRISPRa works
                st.subheader("🧪 How CRISPR Works")
                st.markdown("""
                **CRISPR Gene Editing Overview:**
                - **CRISPRi (CRISPR Interference):** Uses a modified **dCas9-KRAB** protein to **block transcription** of target genes, effectively silencing them. This approach is used to suppress oncogenes (genes that promote cancer growth).
                - **CRISPRa (CRISPR Activation):** Uses **dCas9-VP64** to **boost transcription** of target genes. This method is useful for reactivating tumor suppressor genes that help prevent cancer.
                - **Guide RNA (gRNA):** A small RNA sequence guides CRISPR to the specific gene of interest.
                - **Delivery Methods:** CRISPR can be delivered using **viral vectors, lipid nanoparticles, or electroporation** depending on the application.
                """)


else:
    st.warning("⚠ Please ensure the trained model files are available to proceed.")


Writing app.py


In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Streamlit App Title
st.title("🧬 Personalized CRISPR Therapy Model")
st.write("Upload your breast cancer gene expression dataset to analyze and get CRISPR-based recommendations.")

# Step 1: Load Both Models & Feature Names
@st.cache_resource
def load_models():
    try:
        with open("trained_model_optimized.pkl", "rb") as model_file:
            model_data_main = pickle.load(model_file)
            main_model = model_data_main["model"]
            main_feature_names = model_data_main["feature_names"]
        with open("scaler_optimized.pkl", "rb") as scaler_file:
            main_scaler = pickle.load(scaler_file)
        with open("label_encoder_optimized.pkl", "rb") as label_encoder_file:
            main_label_encoder = pickle.load(label_encoder_file)

        with open("trained_model_basal_luminalB.pkl", "rb") as model_file2:
            model_data_second = pickle.load(model_file2)
            second_model = model_data_second["model"]
            second_feature_names = model_data_second["feature_names"]
        with open("scaler_basal_luminalB.pkl", "rb") as scaler_file2:
            second_scaler = pickle.load(scaler_file2)
        with open("label_encoder_basal_luminalB.pkl", "rb") as label_encoder_file2:
            second_label_encoder = pickle.load(label_encoder_file2)

        return (
            main_model, main_scaler, main_label_encoder, main_feature_names,
            second_model, second_scaler, second_label_encoder, second_feature_names
        )
    except FileNotFoundError:
        st.error("🚨 Error: Trained model files not found. Please ensure they are uploaded before running the app.")
        return None, None, None, None, None, None, None, None


# Load both models
(
    main_model, main_scaler, main_label_encoder, main_feature_names,
    second_model, second_scaler, second_label_encoder, second_feature_names
) = load_models()

if main_model is not None and second_model is not None:
    st.success("✅ Models Loaded Successfully!")

    # Step 2: Upload a Patient's Gene Expression Data
    st.subheader("🧪 Upload a Patient's Gene Expression Data")
    patient_file = st.file_uploader("Upload Patient CSV", type=["csv"], key="patient_data")

    if patient_file is not None:
        patient_df = pd.read_csv(patient_file)

        if 'type' in patient_df.columns:
            st.error("🚨 Error: The uploaded patient file should not contain a 'type' column.")
        else:
            st.write("Patient Data Preview:")
            st.dataframe(patient_df.head())

            final_prediction = None  # Initialize to prevent NameError

            missing_cols_main = set(main_feature_names) - set(patient_df.columns)
            if missing_cols_main:
                st.error(f"🚨 Error: The uploaded patient file is missing columns for the main model: {missing_cols_main}")
            else:
                patient_df_main = patient_df[main_feature_names]
                patient_scaled_main = main_scaler.transform(patient_df_main)
                probabilities_main = main_model.predict_proba(patient_scaled_main)
                max_confidence = np.max(probabilities_main, axis=1)
                prediction_main = main_model.predict(patient_scaled_main)
                predicted_label_main = main_label_encoder.inverse_transform(prediction_main)

                if max_confidence[0] < 0.5 or predicted_label_main[0] not in ["Normal", "HER2", "Luminal A", "Cell Line"]:
                    st.warning("⚠ Main model is uncertain. Checking second model for Luminal B & Basal...")
                    missing_cols_second = set(second_feature_names) - set(patient_df.columns)
                    if missing_cols_second:
                        st.error("🚨 The second model cannot be used because the patient file is missing required gene expression features.")
                        final_prediction = "Uncertain"
                    else:
                        patient_df_second = patient_df[second_feature_names]
                        patient_scaled_second = second_scaler.transform(patient_df_second)
                        prediction_second = second_model.predict(patient_scaled_second)
                        predicted_label_second = second_label_encoder.inverse_transform(prediction_second)
                        final_prediction = predicted_label_second[0]
                        st.success(f"🩺 Predicted Cancer Subtype: *{final_prediction}*")
                else:
                    final_prediction = predicted_label_main[0]
                    st.success(f"🩺 Predicted Cancer Subtype: *{final_prediction}*")

            if final_prediction == "Uncertain":
                st.error("🚨 The model could not confidently predict a cancer subtype. Please check your uploaded file.")
            else:
                st.subheader("🧬 Recommended CRISPR Gene Alteration")
                if final_prediction in ["HER2", "Luminal A", "Luminal B"]:
                    recommended_action = "Use CRISPRi (Gene Suppression)"
                    st.info(f"🔴 The following overexpressed genes might be contributing to cancer. \n\n*{recommended_action}* to suppress their activity:")
                else:
                    recommended_action = "Use CRISPRa (Gene Activation)"
                    st.info(f"🟢 The following underexpressed genes might be protective. \n\n*{recommended_action}* to boost their activity:")

                top_genes = patient_df.iloc[0].sort_values(ascending=False).head(10)
                affected_genes_df = pd.DataFrame({"Gene": top_genes.index, "Expression Level": top_genes.values})

                if "Probe_ID" in patient_df.columns:
                    affected_genes_df.insert(0, "Probe ID", patient_df["Probe_ID"].iloc[:10].values)
                else:
                    st.warning("⚠ Warning: Probe ID column is missing in the uploaded file. Only gene names will be displayed.")

                st.dataframe(affected_genes_df)

else:
    st.warning("⚠ Please ensure the trained model files are available to proceed.")


2025-02-20 14:20:20.314 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-20 14:20:20.315 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-20 14:20:20.318 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-20 14:20:20.319 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-20 14:20:20.320 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-20 14:20:20.321 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-20 14:20:20.325 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-20 14:20:20.326 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Streamlit App Title
st.title("🧬 Personalized CRISPR Therapy Model")
st.write("Upload your breast cancer gene expression dataset to analyze and get CRISPR-based recommendations.")

# Step 1: Load Pre-trained Model & Feature Names
@st.cache_resource
def load_model():
    try:
        with open("trained_model_optimized.pkl", "rb") as model_file:
            model_data = pickle.load(model_file)
            model = model_data["model"]
            feature_names = model_data["feature_names"]
        with open("scaler_optimized.pkl", "rb") as scaler_file:
            scaler = pickle.load(scaler_file)
        with open("label_encoder_optimized.pkl", "rb") as label_encoder_file:
            label_encoder = pickle.load(label_encoder_file)
        return model, scaler, label_encoder, feature_names
    except FileNotFoundError:
        st.error("🚨 Error: Trained model files not found. Please ensure they are uploaded before running the app.")
        return None, None, None, None

# Load model, scaler, and feature names
model, scaler, label_encoder, feature_names = load_model()

# ✅ Gene Mapping Data (Top 15 Genes for Each Subtype)
gene_info = {
    "HER": [
        {"Gene": "ERBB2", "Effect": "HER2 receptor overexpression", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "GRB7", "Effect": "Growth signaling protein", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "PIK3CA", "Effect": "Signal transduction", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "FOXA1", "Effect": "Transcription factor", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "PTEN", "Effect": "Tumor suppressor loss", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "AKT1", "Effect": "Enhances tumor survival", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "HER3", "Effect": "HER family receptor", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "VEGFA", "Effect": "Angiogenesis activator", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "CDH1", "Effect": "Cell adhesion disruption", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "RB1", "Effect": "Cell cycle checkpoint loss", "CRISPR Recommendation": "Activate with CRISPRa"},
    ],
    "normal": [
        {"Gene": "GATA3", "Function": "Transcription factor", "Effect": "Maintains normal tissue identity", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "BRCA1", "Function": "DNA repair protein", "Effect": "Prevents genetic damage", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "FOXA1", "Function": "Transcription factor", "Effect": "Regulates hormonal genes", "CRISPR Recommendation": "Activate with CRISPRa"},
    ],
    "luminal_A": [
        {"Gene": "ESR1", "Effect": "Estrogen receptor overexpression", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "PGR", "Effect": "Progesterone receptor upregulation", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "GATA3", "Effect": "Transcription factor regulation", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "BRCA2", "Effect": "DNA repair deficiency", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "FOXA1", "Effect": "Regulates estrogen signaling", "CRISPR Recommendation": "Activate with CRISPRa"},
    ],
    "luminal_B": [
        {"Gene": "CCND1", "Effect": "Cell cycle dysregulation", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "FOXA1", "Effect": "Transcription factor", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "MYB", "Effect": "Enhances proliferation", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "BRCA1", "Effect": "DNA repair loss", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "CDK4", "Effect": "Uncontrolled division", "CRISPR Recommendation": "Suppress with CRISPRi"},
    ],
    "basal": [
        {"Gene": "TP53", "Effect": "Tumor suppressor mutation", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "EGFR", "Effect": "Growth factor overexpression", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "RB1", "Effect": "Cell cycle dysregulation", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "BRCA1", "Effect": "DNA repair defect", "CRISPR Recommendation": "Activate with CRISPRa"},
        {"Gene": "MMP9", "Effect": "Metastasis driver", "CRISPR Recommendation": "Suppress with CRISPRi"},
    ],
    "cell_line": [
        {"Gene": "MYC", "Function": "Oncogene", "Effect": "Drives cell proliferation", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "TERT", "Function": "Telomerase enzyme", "Effect": "Maintains unlimited growth", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "CDK4", "Function": "Cell cycle kinase", "Effect": "Promotes rapid division", "CRISPR Recommendation": "Suppress with CRISPRi"},
        {"Gene": "CCNE1", "Function": "Cell cycle regulator", "Effect": "Amplified in aggressive cancers", "CRISPR Recommendation": "Suppress with CRISPRi"},
    ]
}

if model is not None:
    st.success("✅ Model Loaded Successfully!")

    # Step 2: Upload a Patient's Gene Expression Data (Without 'type' column)
    st.subheader("🧪 Upload a Patient's Gene Expression Data")
    patient_file = st.file_uploader("Upload Patient CSV", type=["csv"], key="patient_data")

    if patient_file is not None:
        patient_df = pd.read_csv(patient_file)

        if 'type' in patient_df.columns:
            st.error("🚨 Error: The uploaded patient file should not contain a 'type' column.")
        else:
            st.write("Patient Data Preview:")
            st.dataframe(patient_df.head())

            missing_cols = set(feature_names) - set(patient_df.columns)
            if missing_cols:
                st.error(f"🚨 Error: The uploaded patient file is missing columns: {missing_cols}")
            else:
                patient_df = patient_df[feature_names]

                # Process patient data
                patient_scaled = scaler.transform(patient_df)
                predictions = model.predict(patient_scaled)
                predicted_labels = label_encoder.inverse_transform(predictions)

                # Add predictions to dataset
                patient_df["Predicted Subtype"] = predicted_labels
                st.subheader("🩺 Predicted Cancer Subtypes")
                st.dataframe(patient_df[["Predicted Subtype"]])

                # Display affected genes for each detected subtype
                unique_subtypes = np.unique(predicted_labels)
                for subtype in unique_subtypes:
                    st.subheader(f"🧬 Affected Genes in {subtype}")
                    if subtype in gene_info:
                        affected_genes_df = pd.DataFrame(gene_info[subtype])
                        st.dataframe(affected_genes_df)
                    else:
                        st.warning(f"⚠ No specific gene information available for {subtype}.")

                # CRISPR Explanation
                st.subheader("🧪 How CRISPR Works")
                st.markdown("""
                - *CRISPRi (CRISPR Interference):* Silences oncogenes by blocking transcription.
                - *CRISPRa (CRISPR Activation):* Reactivates tumor suppressor genes.
                - *Delivery Methods:* Viral vectors, lipid nanoparticles, electroporation.
                """)

else:
    st.warning("⚠ Please ensure the trained model files are available to proceed.")

Writing app.py


In [ ]:
pip install streamlit


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.9 MB/s eta 0:00:00


In [ ]:
!pip install streamlit pyngrok


In [ ]:
!ngrok authtoken 2tLVt7HlK8X6Qfz7AaADkqXadjQ_7JvvycodZMNGRpbYDED7u


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok

# Start the tunnel for port 8501
public_url = ngrok.connect(8501, "http")
print("Streamlit App URL:", public_url)

Streamlit App URL: NgrokTunnel: "https://9db6-34-75-217-3.ngrok-free.app" -> "http://localhost:8501"


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴

⠦⠧⠇Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.75.217.3:8501

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [ ]:
pip install scikit-learn==1.1.3


  Using cached scikit_learn-1.1.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
Using cached scikit_learn-1.1.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (32.0 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
imbalanced-learn 0.13.0 requires scikit-learn<2,>=1.3.2, but you have scikit-learn 1.1.3 which is incompatible.
mlxtend 0.23.4 requires scikit-learn>=1.3.1, but you have scikit-learn 1.1.3 which is incompatible.
bigframes 1.36.0 requires scikit-learn>=1.2.2, but you have scikit-learn 1.1.3 which is incompatible.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.1.3 which is incompatible.


In [ ]:
!pip install --upgrade scikit-learn imbalanced-learn

  Using cached scikit_learn-1.6.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (18 kB)
Using cached scikit_learn-1.6.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.5 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.1.3
    Uninstalling scikit-learn-1.1.3:
      Successfully uninstalled scikit-learn-1.1.3


In [ ]:
import numpy as np
import pandas as pd
import pickle

# Load the trained model
model_file_path = "/content/trained_model_optimized.pkl"
with open(model_file_path, "rb") as model_file:
    model_data = pickle.load(model_file)

# Extract feature names
expected_features = model_data["feature_names"]

# Generate test patient data
num_samples = 5
np.random.seed(42)
test_patient_data = np.random.uniform(6, 12, size=(num_samples, len(expected_features)))

# Create DataFrame with correct feature names
test_patient_df = pd.DataFrame(test_patient_data, columns=expected_features)

# Save dataset
test_patient_df.to_csv("/content/Test_Patient_Dataset.csv", index=False)
